# Módulo F4 · Matemáticas para ML — Práctica guiada

**Caso:** *TiendaSur* quiere entender a sus clientes. Dirección pregunta:

1. ¿Qué clientes se parecen entre sí? *(distancias y similitud coseno)*
2. ¿Qué variables del dataset van juntas? *(covarianza)*
3. ¿Podemos "aprender" el gasto esperado en función de los pedidos con un modelo a mano? *(gradiente descendente)*
4. Si un cliente abre el email, ¿cuánto sube la probabilidad de que compre? *(Bayes)*

Todo se responde con NumPy, sin librerías de ML: la idea es ver la maquinaria funcionando a mano.

> **Ejecuta las celdas en orden.** Todas se pueden ejecutar en local o Colab sin descargar datos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

In [ ]:
# Celda de setup: reproduce las definiciones necesarias del notebook del módulo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

clientes = pd.DataFrame({
    "cliente":      ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo"],
    "gasto":        [120,  480,  90,  150, 510, 300],
    "pedidos":      [4,    20,   3,    5,   18,  10],
    "ticket_medio": [30,   24,   30,   30,  28,  30],
})
X = clientes[["gasto", "pedidos", "ticket_medio"]].to_numpy(float)

def distancia_euclidea(a, b):
    return np.linalg.norm(a - b)

def similitud_coseno(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

def derivada_numerica(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

pedidos_mensuales = np.array([ 2,  3,  4,  5,  6,  7,  8,  9, 10])
gasto_mensual     = np.array([45, 60, 80, 95,110,125,150,165,180])

# Escenario de Bayes
P_abre, P_compra, P_compra_dado_abre = 0.10, 0.05, 0.40
P_no_abre = 1 - P_abre
P_compra_dado_no = (P_compra - P_compra_dado_abre * P_abre) / P_no_abre
print("setup listo")

# Soluciones del módulo F4

## E1 · Vecino más parecido (coseno vs. distancia)

In [ ]:
nombres = clientes["cliente"].to_numpy()
for i in range(len(X)):
    cosenos = [similitud_coseno(X[i], X[j]) if i != j else -2 for j in range(len(X))]
    dists   = [distancia_euclidea(X[i], X[j]) if i != j else 1e9 for j in range(len(X))]
    vecino_cos = nombres[int(np.argmax(cosenos))]
    vecino_dis = nombres[int(np.argmin(dists))]
    print(f"{nombres[i]:>8}: por coseno → {vecino_cos:<8} | por distancia → {vecino_dis:<8}")

## E2 · Covarianza con edad

In [ ]:
X2 = np.column_stack([X, [25, 45, 19, 32, 50, 40]])
labels2 = ["gasto", "pedidos", "ticket_medio", "edad"]
cov2 = np.cov(X2.T)
for i in range(4):
    for j in range(i + 1, 4):
        print(f"cov({labels2[i]:>12}, {labels2[j]:<12}) = {cov2[i, j]:9.2f}")

## E3 · Descenso con tres tasas de aprendizaje

In [ ]:
def entrenar(lr, pasos=500):
    w, b = 0.0, 0.0
    errores = []
    for _ in range(pasos):
        y_pred = w * pedidos_mensuales + b
        error = y_pred - gasto_mensual
        errores.append(float(np.mean(error**2)))
        grad_w = 2 / len(gasto_mensual) * np.sum(error * pedidos_mensuales)
        grad_b = 2 / len(gasto_mensual) * np.sum(error)
        w -= lr * grad_w
        b -= lr * grad_b
    return errores

for lr, etiqueta in [(0.00005, "0.00005 (lento)"), (0.001, "0.001 (OK)"), (0.05, "0.05 (grande)")]:
    plt.plot(entrenar(lr), label=etiqueta)
plt.yscale("log"); plt.legend(); plt.xlabel("paso"); plt.ylabel("MSE")
plt.title("Descenso con tres tasas de aprendizaje"); plt.show()

**Lectura:** la tasa lenta apenas avanza en 500 pasos; la media converge con suavidad; la grande oscila o diverge. La tasa correcta depende de la escala de los datos — en ML real se normaliza por esto.

## E4 · descenso() genérico sobre f(x) = x² − 4x + 4

In [ ]:
def descenso(f, x0, lr, pasos=200):
    x = float(x0)
    for _ in range(pasos):
        x -= lr * derivada_numerica(f, x)
    return x

f_e4 = lambda x: x**2 - 4*x + 4        # mínimo analítico: x=2, f=0
xf = descenso(f_e4, x0=0.0, lr=0.1)
print(f"x ≈ {xf:.4f}, f(x) ≈ {f_e4(xf):.6f}")

## E5 · Campaña mejorada (P(compra | abre) = 0.55)

In [ ]:
P_compra_dado_abre2 = 0.55
P_compra_dado_no2 = (P_compra - P_compra_dado_abre2 * P_abre) / P_no_abre
print(f"P(compra | no abre) = {P_compra_dado_no2:.4f}")

**Respuesta:** sale ≈ −0.006, negativo — imposible como probabilidad. No es un fallo de la fórmula sino de los supuestos: si el 55 % de 10 abridores ya compra, supera el 5 % global; con P(compra)=0.05 y P(abre)=0.10 no hay hueco para el resto. La lección: cuando una fórmula devuelve probabilidad negativa, uno de los supuestos está roto. Para el negocio, mejor razonar sobre ratios de conversión del grupo que abre y conservador sobre el resto.